# CompressRoute: Full GPU Training & Compression Notebook (Colab / Kaggle)

This notebook trains the full **BERT-base Teacher** and **DistilBERT Baseline** on the complete Banking77 dataset, applies **Knowledge Distillation**, exports to **ONNX**, quantizes to **INT8**, and benchmarks all variants.

### Instructions:
1. **Runtime Type**: In Google Colab, select `Runtime` -> `Change runtime type` -> **T4 GPU** (or any GPU).
2. Run all cells in sequence.
3. Expected accuracy on full Banking77 test set: **~92-93%** for BERT-base Teacher.

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Clone repository or pull latest changes
!git clone https://github.com/Dhanuskarthi/Natural-Language-Processing.git
%cd Natural-Language-Processing
!git pull

In [ ]:
# Install pinned dependencies
!pip install -q -r requirements.txt

## 1. Full Teacher & Baseline Fine-Tuning (Banking77)
Trains `bert-base-uncased` (Teacher) and `distilbert-base-uncased` (Baseline) for 3-4 epochs with lr=2e-5 and batch size 32.

In [ ]:
!python -m compress.finetune_teacher --model_type all

## 2. Knowledge Distillation (Phase 2)
Distills BERT-base teacher knowledge into a compact 4-layer student using combined Cross-Entropy and KL-divergence loss ($T=3.0, \alpha=0.5$).

In [ ]:
!python -m compress.distill

## 3. ONNX FP32 Export & INT8 Quantization (Phase 3)
Exports PyTorch checkpoints to ONNX graphs and applies dynamic INT8 quantization with ONNX Runtime.

In [ ]:
!python -m compress.export_onnx
!python -m compress.quantize

## 4. Benchmark All Variants (Phase 4)
Measures latency (p50, p95, p99), throughput, and memory footprint across batch sizes and sequence lengths.

In [ ]:
!python -m benchmark.run_benchmark

## 5. Download Results
Download the updated `benchmark/results/registry.json` and metric files to commit to your GitHub repository.

In [ ]:
from google.colab import files
!tar -czvf results_bundle.tar.gz benchmark/results/*.json
files.download('results_bundle.tar.gz')